# Qualcomm (QCOM) — FY2029 EV/EBITDA Valuation

**Sum-of-the-parts EV/EBITDA**, three scenarios (**Bear / Base / Bull**), discounted to a present value.

- **Base = management's June 24, 2026 Investor Day FY29 targets**, taken at face value.
- The whole thesis is that a low-multiple handset/licensing business is being diluted by higher-multiple automotive, IoT, and (especially) **data-center** businesses — so each segment is valued on its own peer set.
- See `README.md` for the company overview, the 2024-vs-2026 Investor Day comparison, and the multiple justifications.

> **Not investment advice.** Every forward number is a management target or an author assumption. Refresh market data before reuse.

**Requirements:** `pip install -r requirements.txt` (pandas, numpy, matplotlib). A zero-dependency version of the same math lives in `model.py`.

## 1. Assumptions

Everything the model needs is in the next cell. Change a number, re-run the notebook, done.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# --- Market anchors (refresh before reuse; gathered ~June 2026) ---
CURRENT_PRICE = 250.0   # QCOM post-Investor-Day (~$222 on Jun 23 pre-day, +~15% pop)
SHARES_OUT    = 1.07    # billion, diluted (FY25)
NET_DEBT      = 1.5     # $B, roughly net-debt-neutral (cash+investments ~ debt)
DISCOUNT_RATE = 0.10    # required annual return
YEARS_TO_FY29 = 3.25    # ~mid-2026 -> FY29 year-end (Sept 2029)

FY25_REVENUE = 44.3     # $B actual
FY25_EBITDA  = 14.0     # $B actual

SEGMENTS = ['handset', 'automotive', 'iot', 'datacenter']

# FY29 revenue ($B). BASE = management Investor-Day targets at face value.
REVENUE = {
    'Bear': {'handset': 18, 'automotive':  8, 'iot': 11, 'datacenter':  7},
    'Base': {'handset': 20, 'automotive': 10, 'iot': 14, 'datacenter': 15},
    'Bull': {'handset': 22, 'automotive': 12, 'iot': 16, 'datacenter': 20},
}
# FY29 EBITDA margins (consistent with QCT ~30% / QTL ~70% op-margin targets + D&A addback)
EBITDA_MARGIN = {
    'Bear': {'handset': 0.35, 'automotive': 0.24, 'iot': 0.26, 'datacenter': 0.18},
    'Base': {'handset': 0.38, 'automotive': 0.28, 'iot': 0.30, 'datacenter': 0.25},
    'Bull': {'handset': 0.40, 'automotive': 0.30, 'iot': 0.33, 'datacenter': 0.30},
}
# FY29 exit EV/EBITDA multiple per segment (peer justification in README §4)
EXIT_MULTIPLE = {
    'Bear': {'handset':  7, 'automotive': 10, 'iot': 10, 'datacenter': 10},
    'Base': {'handset':  9, 'automotive': 13, 'iot': 14, 'datacenter': 18},
    'Bull': {'handset': 11, 'automotive': 16, 'iot': 18, 'datacenter': 25},
}
WEIGHTS = {'Bear': 0.25, 'Base': 0.50, 'Bull': 0.25}

print(f'Trailing EV/EBITDA on FY25: ~{(CURRENT_PRICE*SHARES_OUT+NET_DEBT)/FY25_EBITDA:.1f}x')

## 2. The engine — segment SOTP → FY29 EV → PV today

For each scenario and segment: `EBITDA = revenue × margin`, `EV = EBITDA × exit multiple`. Sum to enterprise value, subtract net debt for FY29 equity value, then discount to present and divide by current shares.

In [ ]:
def segment_table(name):
    df = pd.DataFrame({'segment': SEGMENTS})
    df['revenue_$B']   = df.segment.map(REVENUE[name])
    df['ebitda_margin']= df.segment.map(EBITDA_MARGIN[name])
    df['ebitda_$B']    = df['revenue_$B'] * df['ebitda_margin']
    df['exit_x']       = df.segment.map(EXIT_MULTIPLE[name])
    df['EV_$B']        = df['ebitda_$B'] * df['exit_x']
    return df

def value_scenario(name):
    df = segment_table(name)
    ev = df['EV_$B'].sum()
    ebitda = df['ebitda_$B'].sum()
    equity_fy29 = ev - NET_DEBT
    price_fy29 = equity_fy29 / SHARES_OUT
    pv_price = (equity_fy29 / (1 + DISCOUNT_RATE) ** YEARS_TO_FY29) / SHARES_OUT
    return {
        'scenario': name,
        'FY29 rev $B': df['revenue_$B'].sum(),
        'FY29 EBITDA $B': round(ebitda, 1),
        'blended x': round(ev / ebitda, 1),
        'FY29 EV $B': round(ev, 0),
        'FY29 price': round(price_fy29, 0),
        'PV today': round(pv_price, 0),
        'upside': pv_price / CURRENT_PRICE - 1,
        'CAGR 3.25y': (price_fy29 / CURRENT_PRICE) ** (1 / YEARS_TO_FY29) - 1,
    }

summary = pd.DataFrame([value_scenario(s) for s in ['Bear', 'Base', 'Bull']]).set_index('scenario')
summary_fmt = summary.copy()
summary_fmt['upside'] = (summary['upside'] * 100).round(0).astype(int).astype(str) + '%'
summary_fmt['CAGR 3.25y'] = (summary['CAGR 3.25y'] * 100).round(0).astype(int).astype(str) + '%'
summary_fmt

In [ ]:
# Probability-weighted fair value (PV today)
blended = sum(WEIGHTS[s] * summary.loc[s, 'PV today'] for s in WEIGHTS)
print(f'Probability-weighted PV (25/50/25): ${blended:.0f}/sh  '
      f'({blended/CURRENT_PRICE-1:+.0%} vs ${CURRENT_PRICE:.0f})')
print(f'At ${CURRENT_PRICE:.0f}, that is ~{CURRENT_PRICE/18:.1f}x the mgmt FY29 EPS target (>$18).')

## 3. Sum-of-the-parts detail (per scenario)

In [ ]:
for s in ['Bear', 'Base', 'Bull']:
    print(f'\n===== {s} =====')
    t = segment_table(s).copy()
    t['ebitda_margin'] = (t['ebitda_margin'] * 100).round(0).astype(int).astype(str) + '%'
    t['ebitda_$B'] = t['ebitda_$B'].round(2)
    print(t.to_string(index=False))

## 4. Charts

In [ ]:
# PV today per share by scenario vs current price
fig, ax = plt.subplots(figsize=(8, 5))
colors = {'Bear': '#c0392b', 'Base': '#7f8c8d', 'Bull': '#27ae60'}
bars = ax.bar(summary.index, summary['PV today'], color=[colors[s] for s in summary.index])
ax.axhline(CURRENT_PRICE, color='black', linestyle='--', linewidth=1)
ax.text(2.4, CURRENT_PRICE + 4, f'price ${CURRENT_PRICE:.0f}', ha='right')
ax.axhline(blended, color='#2980b9', linestyle=':', linewidth=1.5)
ax.text(-0.45, blended + 4, f'prob-wtd ${blended:.0f}', color='#2980b9')
for b, v in zip(bars, summary['PV today']):
    ax.text(b.get_x() + b.get_width()/2, v + 4, f'${v:.0f}', ha='center', fontweight='bold')
ax.set_ylabel('PV today per share ($)')
ax.set_title('QCOM intrinsic value (PV today) by scenario vs. market price')
plt.tight_layout(); plt.show()

In [ ]:
# Segment EV contribution (FY29) — stacked by scenario
ev_by_seg = pd.DataFrame({s: segment_table(s).set_index('segment')['EV_$B'] for s in ['Bear','Base','Bull']}).T
ax = ev_by_seg.plot(kind='bar', stacked=True, figsize=(8, 5),
                    color=['#34495e', '#2980b9', '#16a085', '#e67e22'])
ax.set_ylabel('FY29 Enterprise Value ($B)')
ax.set_title('Where the value comes from: segment EV by scenario')
ax.legend(title='segment', bbox_to_anchor=(1.01, 1))
plt.xticks(rotation=0); plt.tight_layout(); plt.show()
print('Data-center share of FY29 EV:',
      {s: f"{ev_by_seg.loc[s,'datacenter']/ev_by_seg.loc[s].sum():.0%}" for s in ev_by_seg.index})

## 5. Sensitivity

The valuation is dominated by the **discount rate** and the **data-center exit multiple**. Both tables below flex the **Base** scenario.

In [ ]:
# Base FY29 equity value, fixed; vary discount rate -> PV today / share
base_ev = segment_table('Base')['EV_$B'].sum()
base_equity = base_ev - NET_DEBT
rates = [0.06, 0.08, 0.10, 0.12, 0.14]
pv = {f'{r:.0%}': round((base_equity / (1+r)**YEARS_TO_FY29) / SHARES_OUT, 0) for r in rates}
pd.DataFrame.from_dict(pv, orient='index', columns=['Base PV today / share ($)']).rename_axis('discount rate')

In [ ]:
# 2-D sensitivity: data-center exit multiple (rows) x discount rate (cols) -> Base PV today / share
dc_mults = [8, 12, 16, 18, 22, 26]
base = segment_table('Base').set_index('segment')
ex_dc_ev = (base['EV_$B'].sum() - base.loc['datacenter', 'EV_$B'])  # EV of non-DC segments
dc_ebitda = base.loc['datacenter', 'ebitda_$B']
grid = {}
for r in rates:
    grid[f'{r:.0%}'] = []
    for m in dc_mults:
        ev = ex_dc_ev + dc_ebitda * m
        grid[f'{r:.0%}'].append(round(((ev - NET_DEBT)/(1+r)**YEARS_TO_FY29)/SHARES_OUT, 0))
sens = pd.DataFrame(grid, index=[f'{m}x' for m in dc_mults]).rename_axis('data-center exit x')
print('Base-case PV today / share ($) — rows: data-center multiple, cols: discount rate')
print(f'(current price ${CURRENT_PRICE:.0f})')
sens

## 6. Investor Day comparison — Nov 2024 vs. June 2026 (FY29 targets)

| FY2029 target | Nov 2024 | June 2026 | Change |
|---|---:|---:|:--|
| **Non-handset revenue** | **$22B** | **$40B** | **+82%** |
| Automotive | $8B | $10B | +25% |
| IoT (total) | $14B | $14B | flat (re-segmented) |
| Data center | *not a stated FY29 line* | **>$15B** | **new pillar** |
| Non-GAAP EPS | ~$18 | >$18 | maintained |
| Mix goal | 50/50 handset vs auto+IoT (2030) | ~⅓ / ⅓ / ⅓ (handset / auto+IoT / data center) | data center added |
| QCT / QTL op margin | 30% / 70% | 30% / 70% | reaffirmed |

The near-doubling of non-handset revenue ($22B→$40B) is almost entirely the **new ~$15B data-center business**. Note EPS barely moved (~$18 → >$18) despite +$18B of revenue — implying lower incremental margin on data center, which is why its **exit multiple** is the swing variable above.

## Takeaway

At ~$250 (post-Investor-Day, +15%), the market already prices QCOM close to the **Bull** case. Management's own FY29 targets, valued on segment-appropriate multiples and discounted at 10%, imply a base intrinsic value well below the current price. The bet you're making at today's price is essentially a bet that the **data-center business lands at or above plan** — size the position to that single risk.